# 데이터 로드

In [7]:
# === pairs 생성 ===
ZIP_PATH = "/content/NIKL_DIALOGUE_2024_v1.0.zip"   # 경로
import zipfile, json, io, re, random

# 옵션
PAIRING_MODE = "next_sentence"
USE_ONLY_ORIGINAL = True         # True: original_form , False: form
MIN_LEN, MAX_LEN = 1, 160        # 텍스트 길이 필터
SHUFFLE = True

def clean(s: str) -> str:
    # 최소 정제: 제로폭/중복공백 제거
    s = s.replace("\u200b", "").strip()
    s = re.sub(r"\s+", " ", s)
    return s

def is_korean(s: str) -> bool:
    kor = len(re.findall(r"[가-힣]", s))
    return kor >= max(2, int(0.2 * len(s)))

pairs = []
n_files = 0
with zipfile.ZipFile(ZIP_PATH) as zf:
    for name in zf.namelist():
        if not name.lower().endswith(".json"):
            continue
        n_files += 1
        with zf.open(name) as f:
            data = json.load(io.TextIOWrapper(f, encoding="utf-8"))
        docs = data.get("document", [])
        for doc in docs:
            utts = doc.get("utterance", [])
            utts = sorted(utts, key=lambda u: (u.get("start", float("inf")), u.get("id", "")))
            seq = []
            for u in utts:
                txt = (u.get("form") or "") if USE_ONLY_ORIGINAL else (u.get("original_form") or u.get("form") or "")
                txt = clean(txt)
                if not txt:
                    continue
                if len(txt) < MIN_LEN or len(txt) > MAX_LEN:
                    continue
                if not is_korean(txt):
                    continue
                seq.append((txt, u.get("speaker_id", None)))

            if len(seq) >= 2:
                if PAIRING_MODE == "turn_change":
                    for (a, sid_a), (b, sid_b) in zip(seq, seq[1:]):
                        if sid_a != sid_b and a and b:
                            pairs.append((a, b))
                else:
                    for (a, _), (b, _) in zip(seq, seq[1:]):
                        if a and b:
                            pairs.append((a, b))

if SHUFFLE:
    random.shuffle(pairs)

print(f"JSON files read: {n_files}")
print(f"Total pairs: {len(pairs)}")
print("Sample:", pairs[0] if pairs else ("<empty>", "<empty>"))


JSON files read: 3227
Total pairs: 720859
Sample: ('생각보다 그 반려동물 입양이 되게 인연인 거 같고', '사랑이 많이 필요한 계기가 있는 거 같아요.')


# 토크나이저

In [8]:
# sentencepiece tokenizatoin

import sentencepiece as spm

with open("corpus.txt", "w", encoding="utf-8") as f:
    for q, a in pairs:
        f.write(q.replace("\n"," ") + " <sep> " + a.replace("\n"," ") + "\n")

spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix="spm16k",
    vocab_size=16000,
    model_type="unigram",
    character_coverage=0.9998,
    pad_id=0, bos_id=1, eos_id=3, unk_id=4,
    control_symbols=["<sep>"],
    byte_fallback=True
)

sp = spm.SentencePieceProcessor(model_file="spm16k.model")

SPECIALS = ["<pad>","<bos>","<sep>","<eos>","<unk>"]
PAD = sp.pad_id(); BOS = sp.bos_id(); EOS = sp.eos_id(); UNK = sp.unk_id()
SEP = sp.piece_to_id("<sep>")
VOCAB_SIZE = sp.get_piece_size()

def encode_text(s: str):
    return sp.encode(s, out_type=int)

def decode_text(ids):
    drop = {PAD, BOS, SEP, EOS, UNK}
    return sp.decode([i for i in ids if i not in drop])

In [9]:
sp = spm.SentencePieceProcessor(model_file="spm16k.model")

SPECIALS = ["<pad>","<bos>","<sep>","<eos>","<unk>"]
PAD = sp.pad_id(); BOS = sp.bos_id(); EOS = sp.eos_id(); UNK = sp.unk_id()
SEP = sp.piece_to_id("<sep>")
VOCAB_SIZE = sp.get_piece_size()

def encode_text(s: str):
    return sp.encode(s, out_type=int)

def decode_text(ids):
    drop = {PAD, BOS, SEP, EOS, UNK}
    return sp.decode([i for i in ids if i not in drop])

In [10]:
labeled = []
for ctx, rsp in pairs:
    ids = [BOS] + encode_text(ctx) + [SEP] + encode_text(rsp) + [EOS]
    labels = [-100] * (len([BOS] + encode_text(ctx) + [SEP])) + ids[len([BOS] + encode_text(ctx) + [SEP]):]
    labeled.append((ids, labels))

print(f"Labeled pairs: {len(labeled)}")
print("Sample labeled pair:", labeled[0] if labeled else ("<empty>", "<empty>"))

Labeled pairs: 720859
Sample labeled pair: ([1, 774, 263, 1286, 3232, 274, 303, 10221, 368, 264, 412, 2, 9436, 277, 1061, 3567, 290, 264, 393, 262, 3], [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 9436, 277, 1061, 3567, 290, 264, 393, 262, 3])


In [11]:
tokens_stream, labels_stream = [], []
for x, y in labeled:  # labeled = [(seq, labels), ...]
    tokens_stream.extend(x)
    labels_stream.extend(y)
    labels_stream[-1] = -100
PACK_LEN = MAX_LEN
PAD_ID = PAD
packed_inputs, packed_labels, packed_attn = [], [], []

N = len(tokens_stream)
for i in range(0, N, PACK_LEN):
    j = min(i + PACK_LEN, N)
    inp = tokens_stream[i:j]
    lbl = labels_stream[i:j]
    real_len = len(inp)
    pad_n = PACK_LEN - real_len
    if pad_n > 0:
        inp = inp + [PAD_ID] * pad_n
        lbl = lbl + [-100] * pad_n
    attn = [1] * real_len + [0] * pad_n

    packed_inputs.append(inp)
    packed_labels.append(lbl)
    packed_attn.append(attn)

import torch
from torch.utils.data import TensorDataset, DataLoader

packed_ds = TensorDataset(
    torch.tensor(packed_inputs, dtype=torch.long),
    torch.tensor(packed_labels, dtype=torch.long),
    torch.tensor(packed_attn, dtype=torch.long),
)

def collate_packed(batch):
    inp  = torch.stack([b[0] for b in batch])
    lbl  = torch.stack([b[1] for b in batch])
    attn = torch.stack([b[2] for b in batch])
    return {"input_ids": inp, "labels": lbl, "attention_mask": attn}

BATCH_SIZE = 128
train_cut = int(len(packed_ds) * 0.95)
train_dl = DataLoader(torch.utils.data.Subset(packed_ds, range(train_cut)),
                      batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                      collate_fn=collate_packed)
val_dl   = DataLoader(torch.utils.data.Subset(packed_ds, range(train_cut, len(packed_ds))),
                      batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
                      collate_fn=collate_packed)

# 트렌스포머

In [12]:
import math, torch.nn as nn, torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.nh = n_heads
        self.dk = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x); q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.nh, self.dk).transpose(1,2)
        k = k.view(B, T, self.nh, self.dk).transpose(1,2)
        v = v.view(B, T, self.nh, self.dk).transpose(1,2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)
        causal = torch.triu(torch.full((T, T), float("-inf"), device=x.device), diagonal=1)
        att = att + causal
        if attn_mask is not None:
            pad_mask = (attn_mask == 0).unsqueeze(1).unsqueeze(2)
            att = att.masked_fill(pad_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = att @ v
        y = y.transpose(1,2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn= CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_ratio*d_model, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x, attn_mask=None):
        x = x + self.attn(self.ln1(x), attn_mask)
        x = x + self.mlp(self.ln2(x))
        return x

class GPTScratch(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=512, n_layers=8, n_heads=8,
                 max_len=MAX_LEN, dropout=0.1):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, 4, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, x, attention_mask=None):
        B, T = x.shape
        if T > self.max_len:
            x = x[:, -self.max_len:]; T = x.shape[1]
            if attention_mask is not None:
                attention_mask = attention_mask[:, -self.max_len:]
        pos = torch.arange(0, T, device=x.device).unsqueeze(0)
        h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        for blk in self.blocks: h = blk(h, attention_mask)
        h = self.ln_f(h)
        return self.head(h)

In [13]:
def count_tokens_pairs(pairs, max_len=160):
    tot = 0
    for ctx, rsp in pairs:
        ids = [257] + encode_text(ctx) + [258] + encode_text(rsp) + [259]  # BOS/SEP/EOS
        tot += min(len(ids), max_len)
    return tot

def count_tokens_mono(docs, max_len=160):
    tot = 0
    for s in docs:
        ids = [257] + encode_text(s) + [259]
        tot += min(len(ids), max_len)
    return tot

TOK_PER_EPOCH = 0
if 'pairs' in globals() and len(pairs) > 0:
    TOK_PER_EPOCH += count_tokens_pairs(pairs, max_len=MAX_LEN)
if 'mono_docs' in globals() and len(mono_docs) > 0:
    TOK_PER_EPOCH += count_tokens_mono(mono_docs, max_len=MAX_LEN)

print(f"≈ tokens per epoch: {TOK_PER_EPOCH:,}")


≈ tokens per epoch: 15,750,093


# 학습

In [ ]:
import torch, torch.nn.functional as F, time, json, os
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GPTScratch(
    vocab_size=VOCAB_SIZE,
    d_model=512, n_layers=8, n_heads=8,
    max_len=MAX_LEN, dropout=0.1
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

def lm_loss(logits, targets):
    vocab = logits.size(-1)
    pred = logits[:, :-1].contiguous().view(-1, vocab)
    gold = targets[:, 1:].contiguous().view(-1)
    return F.cross_entropy(pred, gold, ignore_index=-100)

@torch.no_grad()
def evaluate():
    model.eval(); tot, n = 0.0, 0
    for b in val_dl:
        x = b["input_ids"].to(device); y = b["labels"].to(device); attn = b["attention_mask"].to(device)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            loss = lm_loss(model(x, attention_mask=attn), y)
        tot += loss.item(); n += 1
    return tot/max(1,n)

EPOCHS, ACCUM = 9, 4    # EPOCH
best = float("inf"); save_dir = "molu_chatbot"

for ep in range(1, EPOCHS+1):
    model.train(); t0=time.time(); optimizer.zero_grad(set_to_none=True); step=0
    for i, b in enumerate(train_dl, 1):
        x = b["input_ids"].to(device); y = b["labels"].to(device); attn = b["attention_mask"].to(device)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            loss = lm_loss(model(x, attention_mask=attn), y) / ACCUM
        scaler.scale(loss).backward()
        if i % ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); step += 1
            if step % 100 == 0: print(f"step {step} | loss {(loss.item()*ACCUM):.4f}")

    val = evaluate(); print(f"[epoch {ep}] val_loss={val:.4f} | {(time.time()-t0)/60:.1f} min")
    if val < best:
        best = val
        os.makedirs(save_dir, exist_ok=True)
        torch.save(model.state_dict(), f"{save_dir}/model.pt")
        with open(f"{save_dir}/config.json","w") as f:
            json.dump({
                "vocab_size": VOCAB_SIZE, "d_model":512,"n_layers":8,"n_heads":8,
                "max_len":MAX_LEN,"dropout":0.1,
                "special_tokens":{"PAD":PAD,"BOS":BOS,"SEP":SEP,"EOS":EOS}
            }, f, indent=2)
        print("SAVED:", save_dir)

step 100 | loss 6.7766
[epoch 1] val_loss=6.3587 | 5.7 min
SAVED: molu_chatbot
step 100 | loss 6.2117
[epoch 2] val_loss=5.9733 | 5.8 min
SAVED: molu_chatbot
step 100 | loss 5.8538
[epoch 3] val_loss=5.6653 | 5.8 min
SAVED: molu_chatbot
step 100 | loss 5.6060
[epoch 4] val_loss=5.4412 | 5.9 min
SAVED: molu_chatbot


# 인퍼런스

In [ ]:
import re, json, torch

SAVE_DIR = "molu_chatbot"  # 저장
device = "cuda" if torch.cuda.is_available() else "cpu"

# state_dict와 config 읽기
sd = torch.load(f"{SAVE_DIR}/model.pt", map_location="cpu")
with open(f"{SAVE_DIR}/config.json") as f:
    cfg = json.load(f)

# state_dict에서 실제 아키텍처 추론
def infer_arch_from_state_dict(state_dict):
    d_model = state_dict["tok_emb.weight"].shape[1]
    vocab_size = state_dict["tok_emb.weight"].shape[0]
    max_len = state_dict["pos_emb.weight"].shape[0]
    layer_idx = set()
    for k in state_dict.keys():
        m = re.match(r"blocks\.(\d+)\.", k)
        if m:
            layer_idx.add(int(m.group(1)))
    n_layers = (max(layer_idx) + 1) if layer_idx else cfg.get("n_layers", 6)
    return vocab_size, d_model, max_len, n_layers

vocab_size, d_model_sd, max_len_sd, n_layers_sd = infer_arch_from_state_dict(sd)

# n_heads 보정
def pick_heads(d_model, preferred=cfg.get("n_heads", 6)):
    divisors = [h for h in range(2, 65) if d_model % h == 0]
    if not divisors:
        raise ValueError(f"d_model={d_model}에 대한 유효한 n_heads가 없습니다.")
    if preferred in divisors:
        return preferred
    return min(divisors, key=lambda h: abs(h - preferred))

n_heads_fixed = pick_heads(d_model_sd, cfg.get("n_heads", 6))

print(f"[info] from state_dict: vocab={vocab_size}, d_model={d_model_sd}, max_len={max_len_sd}, n_layers={n_layers_sd}")
print(f"[info] cfg requested n_heads={cfg.get('n_heads', None)} -> using n_heads={n_heads_fixed}")

# GPTScratch
model = GPTScratch(
    vocab_size=vocab_size,
    d_model=d_model_sd,
    n_layers=n_layers_sd,
    n_heads=n_heads_fixed,   # 헤드 보정
    max_len=max_len_sd,
    dropout=cfg.get("dropout", 0.1),
).to(device).eval()

# 가중치 로드
missing, unexpected = model.load_state_dict(sd, strict=False)
print("loaded. missing:", missing, "unexpected:", unexpected)


[info] from state_dict: vocab=16000, d_model=512, max_len=160, n_layers=8
[info] cfg requested n_heads=8 -> using n_heads=8
loaded. missing: [] unexpected: []


In [ ]:
# === char inference ===
import torch

HIST_MAX = MAX_LEN
MAX_NEW  = 96

@torch.no_grad()
def sample_top_p(logits_row, top_p=0.9, temperature=0.7):
    logits_row = logits_row / max(1e-6, temperature)
    probs = torch.softmax(logits_row, dim=-1)
    sp, si = torch.sort(probs, descending=True)
    cum = torch.cumsum(sp, dim=-1)
    mask = cum > top_p
    mask[..., 0] = False
    sp = sp.masked_fill(mask, 0)
    if sp.sum() <= 0:
        return int(torch.argmax(probs).item())
    sp = sp / sp.sum()
    idx = torch.multinomial(sp, 1)
    return int(si[idx])

@torch.no_grad()
def generate_reply(prompt: str,
                   max_ctx: int = HIST_MAX,
                   max_new_tokens: int = MAX_NEW,
                   top_p=0.9, temperature=0.7):
    history = [BOS] + encode_text(prompt) + [SEP]
    history = history[-max_ctx:]
    x = torch.tensor(history, dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_tokens):
        logits = model(x)
        nxt = sample_top_p(logits[0, -1], top_p=top_p, temperature=temperature)
        history.append(nxt)
        x = torch.tensor(history[-max_ctx:], dtype=torch.long, device=device).unsqueeze(0)
        if nxt == EOS: break
    try:
        last_sep = len(history) - 1 - history[::-1].index(SEP)
    except ValueError:
        last_sep = 0
    try:
        last_eos = len(history) - 1 - history[::-1].index(EOS)
    except ValueError:
        last_eos = len(history)
    return decode_text(history[last_sep:last_eos]).strip()


# 채팅

In [ ]:
print(generate_reply("바다로 여행가면 좋은점이 뭘까?"))

어 네. 맞습니다. 저희 집 근처는 그냥 바다만 가면 그 동네로 인제 그 어우러져 있는 곳이 많이 있었고. 그 정도. 그래서 인제 관광객들이 인제 그런 시설도 많이 있고. 인제 그리고 인제 어 내 나라도 뭐 어 젊은 사람들이 뭐 인제 많이 인제 많이 이제 가잖아요. 뭐 가신 분이에요. 인제 그런 거. 그래서 인제 좀 어 굉장히 좀 어 많이 이제 그 몸에 앞날에 그런 식구 가꾸
